# Qualitative News Prep

Use this notebook when you want to run the qualitative/news prep step-by-step instead of rerunning the full script every time.

The expensive step is filtering `data/raw/nasdaq_exteral_data.csv`. If `data/processed/qualitative/pilot_nasdaq_articles.csv` already exists, load that file and skip the raw CSV scan.

In [14]:
from pathlib import Path
import importlib
import sys

import pandas as pd

repo_root = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "README.md").exists() and (candidate / "processing").exists():
        repo_root = candidate
        break

sys.path.insert(0, str(repo_root / "processing"))

import qual_prep as qp
qp = importlib.reload(qp)

# Use absolute paths so the notebook works from the repo root or from processing/.
qp.INPUT_PATH = repo_root / "data/raw/nasdaq_exteral_data.csv"
qp.TRADING_CALENDAR_PATH = repo_root / "data/processed/trading_calendar.csv"
qp.OUTPUT_DIR = repo_root / "data/processed/qualitative"

processed_path = qp.OUTPUT_DIR / "pilot_nasdaq_articles.csv"
company_day_path = qp.OUTPUT_DIR / "pilot_company_day_news.csv"

# Set to True when preprocessing code changes and you need to regenerate the article table.
REBUILD_ARTICLES = False

qp.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {repo_root}")
print(f"Raw input exists: {qp.INPUT_PATH.exists()} -> {qp.INPUT_PATH}")
print(f"Trading calendar exists: {qp.TRADING_CALENDAR_PATH.exists()} -> {qp.TRADING_CALENDAR_PATH}")

Repo root: c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye
Raw input exists: True -> c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\raw\nasdaq_exteral_data.csv
Trading calendar exists: True -> c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\processed\trading_calendar.csv


In [15]:
calendar = qp.load_trading_calendar()

if calendar is None:
    print("No trading calendar found. Article prep can still run, but company-day output will be skipped.")
else:
    print(calendar.head())
    print(f"Trading days: {len(calendar):,}")

         date                    open_et                   close_et  \
0  2020-01-02  2020-01-02 09:30:00-05:00  2020-01-02 16:00:00-05:00   
1  2020-01-03  2020-01-03 09:30:00-05:00  2020-01-03 16:00:00-05:00   
2  2020-01-06  2020-01-06 09:30:00-05:00  2020-01-06 16:00:00-05:00   
3  2020-01-07  2020-01-07 09:30:00-05:00  2020-01-07 16:00:00-05:00   
4  2020-01-08  2020-01-08 09:30:00-05:00  2020-01-08 16:00:00-05:00   

  early_close close_time  
0       FALSE   16:00:00  
1       FALSE   16:00:00  
2       FALSE   16:00:00  
3       FALSE   16:00:00  
4       FALSE   16:00:00  
Trading days: 2,010


In [16]:
def has_stale_calendar_mapping(articles, calendar):
    if calendar is None or articles.empty:
        return False

    calendar_start = pd.to_datetime(calendar["date"].min())
    article_dates = pd.to_datetime(articles["Date"], errors="coerce", utc=True).dt.tz_localize(None)
    too_old = article_dates < (calendar_start - pd.Timedelta(days=7))
    mapped_to_calendar_start = articles["eligible_date"] == calendar_start.date().isoformat()
    return (too_old & mapped_to_calendar_start).any()

if processed_path.exists() and not REBUILD_ARTICLES:
    articles = pd.read_csv(processed_path, dtype=str)
    if has_stale_calendar_mapping(articles, calendar):
        print("Existing processed file has stale pre-calendar articles mapped to the first trading day. Rebuilding...")
        raw_articles = qp.load_pilot_articles(chunksize=100_000)
        articles = qp.prepare_articles(raw_articles, calendar)
        articles.to_csv(processed_path, index=False)
        print(f"Saved processed articles: {len(articles):,} -> {processed_path}")
    else:
        print(f"Loaded existing processed articles: {len(articles):,}")
else:
    raw_articles = qp.load_pilot_articles(chunksize=100_000)
    articles = qp.prepare_articles(raw_articles, calendar)
    articles.to_csv(processed_path, index=False)
    print(f"Saved processed articles: {len(articles):,} -> {processed_path}")

articles.head()

Existing processed file has stale pre-calendar articles mapped to the first trading day. Rebuilding...
chunk 2: kept 8,959 pilot rows
chunk 15: kept 7,439 pilot rows
chunk 24: kept 8,689 pilot rows
chunk 26: kept 250 pilot rows
chunk 34: kept 3,333 pilot rows
chunk 39: kept 2,612 pilot rows
chunk 58: kept 2 pilot rows
Saved processed articles: 31,284 -> c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\processed\qualitative\pilot_nasdaq_articles.csv


,article_id,Date,eligible_date,Stock_symbol,Article_title,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary,relevance_status,article_scope,article_type,manual_notes
8954,0dca29b4c0f4a9aa,2016-12-12 00:00:00 UTC,,AMD,Why Sirius XM Holdings Inc. (SIRI) Stock Could...,https://www.nasdaq.com/articles/why-sirius-xm-...,NaN,NaN,InvestorPlaceInvestorPlace - Stock Market News...,"Why Advanced Micro Devices, Inc. (AMD) Stock W...","Why Advanced Micro Devices, Inc. (AMD) Stock W...","Why Advanced Micro Devices, Inc. (AMD) Stock W...","Why Advanced Micro Devices, Inc. (AMD) Stock W...",,,,
8955,3ebc90e102bea8b5,2016-12-12 00:00:00 UTC,,AMD,Mid-Day Market Update: Crude Oil Up Over 3%; O...,https://www.nasdaq.com/articles/mid-day-market...,NaN,NaN,"Midway through trading Monday, the Dow traded ...",The two combination trials in wet age-related ...,The two combination trials in wet age-related ...,The two combination trials in wet age-related ...,The two combination trials in wet age-related ...,,,,
8956,ce8396635964bdea,2016-12-12 00:00:00 UTC,,AMD,Value Battle: Pfizer Inc. (PFE) vs. Merck & Co...,https://www.nasdaq.com/articles/value-battle%3...,NaN,NaN,InvestorPlaceInvestorPlace - Stock Market News...,"Should You Buy Advanced Micro Devices, Inc. (A...","Should You Buy Advanced Micro Devices, Inc. (A...","Should You Buy Advanced Micro Devices, Inc. (A...","Should You Buy Advanced Micro Devices, Inc. (A...",,,,
8957,edab41cba28bf3f4,2016-12-12 00:00:00 UTC,,AMD,"Should I Buy AT&T Inc. (T) Stock? 3 Pros, 3 Cons",https://www.nasdaq.com/articles/should-i-buy-a...,NaN,NaN,InvestorPlaceInvestorPlace - Stock Market News...,"Should You Buy Advanced Micro Devices, Inc. (A...","Should You Buy Advanced Micro Devices, Inc. (A...","Should You Buy Advanced Micro Devices, Inc. (A...","Should You Buy Advanced Micro Devices, Inc. (A...",,,,
8958,8f3dfaa9316c233f,2016-12-12 00:00:00 UTC,,AMD,Can Qualcomm Take Intel's Share in the Data Ce...,https://www.nasdaq.com/articles/can-qualcomm-t...,NaN,NaN,"The market leader in mobile chipsets, Qualcomm...","However, one must note that firms like Advance...",Click to get this free report INTL BUS MACH (I...,Click to get this free report INTL BUS MACH (I...,"However, one must note that firms like Advance...",,,,


In [17]:
if calendar is None:
    raise FileNotFoundError(qp.TRADING_CALENDAR_PATH)

company_day_news = qp.build_company_day_news(articles, calendar)
company_day_news.to_csv(company_day_path, index=False)

print(f"Saved company-day news table: {len(company_day_news):,} -> {company_day_path}")
company_day_news.head()

Saved company-day news table: 6,030 -> c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\processed\qualitative\pilot_company_day_news.csv


,Stock_symbol,date,article_ids,article_count,no_news
0,AMD,2020-01-02,"26f737ee359557a6,b7f7bb241bc2e12d,647f7fe0dfe9...",17,False
1,AMD,2020-01-03,"1590e6ca8b4f925f,012b0f8694b72fe5,f81475d5a773...",13,False
2,AMD,2020-01-06,"0c108f2055222bc8,f6d42881b36858cd,2b831379ee15...",10,False
3,AMD,2020-01-07,"52bbcd3a81031109,cb679a3a4cda7a41,6ad86ae909fb...",4,False
4,AMD,2020-01-08,"6fc05944f545bacc,6b15829892ef4f3e,3d106b15967c...",8,False


In [18]:
print("Articles by symbol:")
display(articles["Stock_symbol"].value_counts())

print("Company-day no-news counts:")
display(company_day_news["no_news"].value_counts())

print("Example company-days with news:")
display(company_day_news[company_day_news["article_count"] > 0].head(10))

Articles by symbol:


Stock_symbol
WFC    11301
MRK    10774
AMD     9209
Name: count, dtype: int64

Company-day no-news counts:


no_news
True     3223
False    2807
Name: count, dtype: int64

Example company-days with news:


,Stock_symbol,date,article_ids,article_count,no_news
0,AMD,2020-01-02,"26f737ee359557a6,b7f7bb241bc2e12d,647f7fe0dfe9...",17,False
1,AMD,2020-01-03,"1590e6ca8b4f925f,012b0f8694b72fe5,f81475d5a773...",13,False
2,AMD,2020-01-06,"0c108f2055222bc8,f6d42881b36858cd,2b831379ee15...",10,False
3,AMD,2020-01-07,"52bbcd3a81031109,cb679a3a4cda7a41,6ad86ae909fb...",4,False
4,AMD,2020-01-08,"6fc05944f545bacc,6b15829892ef4f3e,3d106b15967c...",8,False
5,AMD,2020-01-09,"04411ca7cc10bb73,59d4603640d21d22,55aa2c058841...",10,False
6,AMD,2020-01-10,"8fe890e429092655,ff3c0d8e54a5e792,ddb0fe527ee8...",6,False
7,AMD,2020-01-13,"3da838b1b596c64c,32c3d653118f87fb,bdab3086066c...",8,False
8,AMD,2020-01-14,"222a412cc8fa22c9,9b5794603ae02e02,6eec13c8be58...",6,False
9,AMD,2020-01-15,"d1d59ecc28030026,4ef52efb5a5c4092,0ceacfcd4715...",5,False
